In [11]:
import pandas as pd 
import sys 
import numpy as np 

In [14]:
file_path = "../data/raw/張瓊之_給馬老師資料20250925_V4(遺失值不用登錄成-1).xlsx"

demo = pd.read_excel(file_path, sheet_name="demographic")
onset = pd.read_excel(file_path, sheet_name="發病年紀")
mmse = pd.read_excel(file_path, sheet_name="MMSE longitudinal")
blood = pd.read_excel(file_path, sheet_name="blood_data_20250921")
casi = pd.read_excel(file_path, sheet_name="CASI longitudinal")
summary = pd.read_excel(file_path, sheet_name="馬老師回填")

### 處理欄位: 命名欄位

In [15]:
# =============================================================================
# 1. 清理 demographic
# =============================================================================

# 第一列是英文欄位名稱，不是患者資料
demo = demo.iloc[1:].copy()

demo.columns = [
    "patient_id",
    "diagnosis",
    "bio_category",
    "apoe",
    "gender",
    "age_first_mmse",
    "education",
    "hypertension",
    "diabetes",
    "hyperlipidemia"
]

# =============================================================================
# 2. 清理 onset
# =============================================================================

onset = onset.rename(columns={
    "Count Number": "patient_id",
    "age of onset": "age_onset",
    "Biological category": "bio_category_onset"
})

# =============================================================================
# 3. 清理 MMSE
# =============================================================================

mmse = mmse.rename(columns={
    "Count Number": "patient_id",
    "Date": "visit_date",
    "MMSE(numerical)": "mmse",
    "CDR (scale)": "cdr_global",
    "CDR_M (scale)": "cdr_memory",
    "CDR_O (scale)": "cdr_orientation",
    "CDR_J (scale)": "cdr_judgment",
    "CDR_C (scale)": "cdr_community",
    "CDR_H (scale)": "cdr_home_hobbies",
    "CDR_P (scale)": "cdr_personal_care",
    "CDR-SOB(numerical)": "cdr_sob"
})

# =============================================================================
# 4. 清理 CASI
# =============================================================================

casi = casi.rename(columns={
    "Count Number": "patient_id",
    "Date": "casi_date",
    "MENMA10": "casi_mental_manipulation",
    "ATTEN8": "casi_attention",
    "ORIEN18": "casi_orientation",
    "LTM10": "casi_long_term_memory",
    "STM12": "casi_short_term_memory",
    "ABSTR12": "casi_abstraction",
    "DRAW10": "casi_drawing",
    "ANML10": "casi_verbal_fluency",
    "LANG10": "casi_language",
    "Total": "casi_total"
})

# =============================================================================
# 5. 清理 blood
# =============================================================================

blood = blood.rename(columns={
    "Count": "patient_id",
    "Date": "blood_date",
    "HDL-C": "hdl",
    "VLDL-C": "vldl",
    "LDL-C": "ldl",
    "T-Cholesterol": "total_cholesterol",
    "Triglyceride": "triglyceride",
    "AC sugar level": "fasting_glucose",
    "HbA1c": "hba1c"
})

### 統一ID、數值與日期型別

In [16]:
# =============================================================================
# 6. 統一 patient_id
# =============================================================================

dataframes = [demo, onset, mmse, casi, blood]

for df in dataframes:
    df["patient_id"] = pd.to_numeric(
        df["patient_id"],
        errors="coerce"
    ).astype("Int64")

# 移除無法辨識 ID 的資料
for df in dataframes:
    df.dropna(subset=["patient_id"], inplace=True)

# =============================================================================
# 7. 日期轉換
# =============================================================================

mmse["visit_date"] = pd.to_datetime(
    mmse["visit_date"],
    errors="coerce"
)

casi["casi_date"] = pd.to_datetime(
    casi["casi_date"],
    errors="coerce"
)

blood["blood_date"] = pd.to_datetime(
    blood["blood_date"],
    errors="coerce"
)

# =============================================================================
# 8. 數值欄位轉換
# =============================================================================

demo_numeric_cols = [
    "gender",
    "age_first_mmse",
    "education",
    "hypertension",
    "diabetes",
    "hyperlipidemia"
]

onset_numeric_cols = ["age_onset"]

mmse_numeric_cols = [
    "mmse",
    "cdr_global",
    "cdr_memory",
    "cdr_orientation",
    "cdr_judgment",
    "cdr_community",
    "cdr_home_hobbies",
    "cdr_personal_care",
    "cdr_sob"
]

casi_numeric_cols = [
    "casi_mental_manipulation",
    "casi_attention",
    "casi_orientation",
    "casi_long_term_memory",
    "casi_short_term_memory",
    "casi_abstraction",
    "casi_drawing",
    "casi_verbal_fluency",
    "casi_language",
    "casi_total"
]

blood_numeric_cols = [
    "hdl",
    "vldl",
    "ldl",
    "total_cholesterol",
    "triglyceride",
    "fasting_glucose",
    "hba1c"
]

for col in demo_numeric_cols:
    demo[col] = pd.to_numeric(demo[col], errors="coerce")

for col in onset_numeric_cols:
    onset[col] = pd.to_numeric(onset[col], errors="coerce")

for col in mmse_numeric_cols:
    mmse[col] = pd.to_numeric(mmse[col], errors="coerce")

for col in casi_numeric_cols:
    casi[col] = pd.to_numeric(casi[col], errors="coerce")

for col in blood_numeric_cols:
    blood[col] = pd.to_numeric(blood[col], errors="coerce")

In [17]:
def set_outside_range_to_nan(df, column, lower, upper):
    invalid = ~df[column].between(lower, upper) & df[column].notna()
    
    print(
        f"{column}: 發現 {invalid.sum()} 筆超出合理範圍"
    )
    
    df.loc[invalid, column] = np.nan


# MMSE：0–30
set_outside_range_to_nan(mmse, "mmse", 0, 30)

# CDR global：0–3
set_outside_range_to_nan(mmse, "cdr_global", 0, 3)

# CDR-SOB：0–18
set_outside_range_to_nan(mmse, "cdr_sob", 0, 18)

# CASI total：0–100
set_outside_range_to_nan(casi, "casi_total", 0, 100)

# CASI 各分項
casi_ranges = {
    "casi_mental_manipulation": (0, 10),
    "casi_attention": (0, 8),
    "casi_orientation": (0, 18),
    "casi_long_term_memory": (0, 10),
    "casi_short_term_memory": (0, 12),
    "casi_abstraction": (0, 12),
    "casi_drawing": (0, 10),
    "casi_verbal_fluency": (0, 10),
    "casi_language": (0, 10)
}

for col, (lower, upper) in casi_ranges.items():
    set_outside_range_to_nan(casi, col, lower, upper)

# 一般人口學合理範圍
set_outside_range_to_nan(demo, "age_first_mmse", 18, 110)
set_outside_range_to_nan(demo, "education", 0, 30)
set_outside_range_to_nan(onset, "age_onset", 0, 110)

mmse: 發現 0 筆超出合理範圍
cdr_global: 發現 0 筆超出合理範圍
cdr_sob: 發現 0 筆超出合理範圍
casi_total: 發現 0 筆超出合理範圍
casi_mental_manipulation: 發現 0 筆超出合理範圍
casi_attention: 發現 0 筆超出合理範圍
casi_orientation: 發現 0 筆超出合理範圍
casi_long_term_memory: 發現 0 筆超出合理範圍
casi_short_term_memory: 發現 0 筆超出合理範圍
casi_abstraction: 發現 0 筆超出合理範圍
casi_drawing: 發現 0 筆超出合理範圍
casi_verbal_fluency: 發現 0 筆超出合理範圍
casi_language: 發現 0 筆超出合理範圍
age_first_mmse: 發現 0 筆超出合理範圍
education: 發現 4 筆超出合理範圍
age_onset: 發現 0 筆超出合理範圍


#### 有四筆education資料 = -1

### 極端血液數值設定為缺失值

In [18]:
blood_ranges = {
    "hdl": (1, 250),
    "vldl": (1, 200),
    "ldl": (1, 500),
    "total_cholesterol": (20, 700),
    "triglyceride": (1, 2000),
    "fasting_glucose": (1, 1000),
    "hba1c": (2, 25)
}

for col, (lower, upper) in blood_ranges.items():
    set_outside_range_to_nan(blood, col, lower, upper)

hdl: 發現 0 筆超出合理範圍
vldl: 發現 0 筆超出合理範圍
ldl: 發現 0 筆超出合理範圍
total_cholesterol: 發現 0 筆超出合理範圍
triglyceride: 發現 0 筆超出合理範圍
fasting_glucose: 發現 0 筆超出合理範圍
hba1c: 發現 0 筆超出合理範圍


#### fasting_glucose有兩筆低於20 目前先不設定NaN

### 處理同一患者同一天重複 MMSE

In [19]:
duplicate_mmse = mmse[
    mmse.duplicated(
        subset=["patient_id", "visit_date"],
        keep=False
    )
].sort_values(["patient_id", "visit_date"])

print(duplicate_mmse)

      patient_id visit_date  mmse  cdr_global  cdr_memory  cdr_orientation  \
2018         201 2022-09-06  26.0         0.5         0.5              0.0   
2019         201 2022-09-06  26.0         0.5         0.5              0.0   
2291         241 2022-09-05  29.0         0.5         0.5              0.0   
2292         241 2022-09-05  29.0         0.5         0.5              0.0   
2749         318 2022-09-07   8.0         1.0         2.0              2.0   
2750         318 2022-09-07   8.0         1.0         2.0              2.0   
2786         323 2022-09-05   0.0         2.0         3.0              3.0   
2787         323 2022-09-05   0.0         2.0         3.0              3.0   
2982         358 2022-09-05   5.0         1.0         2.0              2.0   
2983         358 2022-09-05   5.0         1.0         2.0              2.0   
3318         415 2022-09-07  20.0         0.5         1.0              0.5   
3319         415 2022-09-07  20.0         0.5         1.0       

### 對mmse long表格同一病人同一日期的資料取平均處理 (可以思考其他方法)

In [20]:
mmse = (
    mmse
    .dropna(subset=["patient_id", "visit_date", "mmse"])
    .groupby(
        ["patient_id", "visit_date"],
        as_index=False
    )[mmse_numeric_cols]
    .mean()
)

### casi和blood資料表沒有同病人同日期的重複資料

In [21]:
duplicate_casi = casi[
    casi.duplicated(
        subset=["patient_id", "casi_date"],
        keep=False
    )
].sort_values(["patient_id", "casi_date"])

print(duplicate_casi)

Empty DataFrame
Columns: [patient_id, casi_date, casi_mental_manipulation, casi_attention, casi_orientation, casi_long_term_memory, casi_short_term_memory, casi_abstraction, casi_drawing, casi_verbal_fluency, casi_language, casi_total]
Index: []


In [22]:
duplicate_blood = blood[
    blood.duplicated(
        subset=["patient_id", "blood_date"],
        keep=False
    )
].sort_values(["patient_id", "blood_date"])

print(duplicate_blood)

Empty DataFrame
Columns: [patient_id, blood_date, hdl, vldl, ldl, total_cholesterol, triglyceride, fasting_glucose, hba1c]
Index: []


### 建立人口學特徵

In [23]:
# 是否攜帶 E4 
# E4 數量
def clean_apoe(value):
    if pd.isna(value):
        return np.nan
    
    value = str(value).upper()
    value = value.replace(" ", "")
    value = value.replace("Ε", "E")
    
    return value


def apoe4_count(value):
    if pd.isna(value):
        return np.nan
    
    value = clean_apoe(value)
    return value.count("E4")


demo["apoe_clean"] = demo["apoe"].apply(clean_apoe)

demo["apoe4_count"] = demo["apoe_clean"].apply(apoe4_count)

demo["apoe4_carrier"] = np.where(
    demo["apoe4_count"].isna(),
    np.nan,
    (demo["apoe4_count"] >= 1).astype(int)
)

### 病程長度

In [24]:
# 把onset的發病年紀併入demo資料表
demo = demo.merge(
    onset[["patient_id", "age_onset"]],
    on="patient_id",
    how="left",
    validate="one_to_one"
)

# 計算疾病持續時間 = baseline MMSE 年紀 - 發病年紀
demo["disease_duration_at_baseline"] = (
    demo["age_first_mmse"] - demo["age_onset"]
)

# 不合理的負值轉為缺失
demo.loc[
    demo["disease_duration_at_baseline"] < 0,
    "disease_duration_at_baseline"
] = np.nan

### 建立 MMSE 歷史軌跡特徵

In [14]:
mmse = mmse.sort_values(
    ["patient_id", "visit_date"]
).reset_index(drop=True)

# 每位患者第幾次 MMSE
mmse["visit_number"] = (
    mmse.groupby("patient_id").cumcount() + 1
)

# 第一次 MMSE 日期
mmse["first_mmse_date"] = (
    mmse.groupby("patient_id")["visit_date"]
    .transform("min")
)

# 距離第一次 MMSE 的年數
mmse["time_since_baseline_years"] = (
    (mmse["visit_date"] - mmse["first_mmse_date"]).dt.days
    / 365.25
)

# 前一次 MMSE
mmse["previous_mmse"] = (
    mmse.groupby("patient_id")["mmse"].shift(1)
)

# 前一次日期
mmse["previous_visit_date"] = (
    mmse.groupby("patient_id")["visit_date"].shift(1)
)

# 與前一次相距時間
mmse["days_since_previous_mmse"] = (
    mmse["visit_date"] - mmse["previous_visit_date"]
).dt.days

# MMSE 最近一次變化
mmse["mmse_change_from_previous"] = (
    mmse["mmse"] - mmse["previous_mmse"]
)

# 年化最近變化率
mmse["recent_mmse_change_per_year"] = (
    mmse["mmse_change_from_previous"]
    / (mmse["days_since_previous_mmse"] / 365.25)
)

In [26]:
mmse.head()

,patient_id,visit_date,mmse,cdr_global,cdr_memory,cdr_orientation,cdr_judgment,cdr_community,cdr_home_hobbies,cdr_personal_care,cdr_sob
0,1,2006-11-13,17.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,6.0
1,1,2007-08-31,15.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,6.0
2,1,2008-01-31,15.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,6.0
3,1,2008-08-14,16.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,5.0
4,1,2009-02-05,17.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,6.0


In [15]:
# 建立歷史平均、標準差、最低值與最高值
grouped_mmse = mmse.groupby("patient_id")["mmse"]

mmse["historical_mmse_mean"] = (
    grouped_mmse.expanding().mean()
    .reset_index(level=0, drop=True)
)

mmse["historical_mmse_std"] = (
    grouped_mmse.expanding().std()
    .reset_index(level=0, drop=True)
)

mmse["historical_mmse_min"] = (
    grouped_mmse.expanding().min()
    .reset_index(level=0, drop=True)
)

mmse["historical_mmse_max"] = (
    grouped_mmse.expanding().max()
    .reset_index(level=0, drop=True)
)

In [16]:
# 建立當下以前的mmse slope
def expanding_slope(group):
    group = group.sort_values("visit_date").copy()
    
    slopes = []
    
    for i in range(len(group)):
        current = group.iloc[:i + 1]
        
        # 至少兩次測量才能估 slope
        if len(current) < 2:
            slopes.append(np.nan)
            continue
        
        x = (
            current["visit_date"] - current["visit_date"].iloc[0]
        ).dt.days.to_numpy() / 365.25
        
        y = current["mmse"].to_numpy()
        
        # 日期完全相同則無法估計
        if np.ptp(x) == 0:
            slopes.append(np.nan)
            continue
        
        slope = np.polyfit(x, y, deg=1)[0]
        slopes.append(slope)
    
    group["historical_mmse_slope"] = slopes
    
    return group


mmse = (
    mmse.groupby("patient_id", group_keys=False)
    .apply(expanding_slope)
    .reset_index(drop=True)
)

C:\Users\User\AppData\Local\Temp\ipykernel_2700\981036836.py:36: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(expanding_slope)


### 建立一年後 MMSE 目標
### 當次 MMSE 後 275–455 天 (9至15個月內)，選擇最接近 365 天的 MMSE，作為一年後目標。

In [20]:
def create_future_mmse_target(
    mmse_df,
    target_days=365,
    min_days=275,
    max_days=455
):
    output_rows = []
    
    for patient_id, group in mmse_df.groupby("patient_id"):
        group = group.sort_values("visit_date").reset_index(drop=True)
        
        for i in range(len(group)):
            current_date = group.loc[i, "visit_date"]
            
            future = group.iloc[i + 1:].copy()
            
            if future.empty:
                continue
            
            future["followup_days"] = (
                future["visit_date"] - current_date
            ).dt.days
            
            future = future[
                future["followup_days"].between(
                    min_days,
                    max_days
                )
            ].copy()
            
            if future.empty:
                continue
            
            # 選最接近一年者
            future["distance_from_target"] = (
                future["followup_days"] - target_days
            ).abs()
            
            target_row = future.loc[
                future["distance_from_target"].idxmin()
            ]
            
            current_row = group.loc[i].to_dict()
            
            current_row["target_date"] = target_row["visit_date"]
            current_row["target_mmse_1y"] = target_row["mmse"]
            current_row["target_followup_days"] = target_row["followup_days"]
            
            # MMSE 實際變化
            current_row["target_mmse_change_1y"] = (
                target_row["mmse"] - group.loc[i, "mmse"]
            )
            
            output_rows.append(current_row)
    
    return pd.DataFrame(output_rows)


model_df = create_future_mmse_target(mmse)

print("一年後預測資料筆數：", len(model_df))
print("患者數：", model_df["patient_id"].nunique())

一年後預測資料筆數： 3193
患者數： 723


### 合併人口學資料

In [21]:
demo_features = [
    "patient_id",
    "diagnosis",
    "bio_category",
    "gender",
    "age_first_mmse",
    "education",
    "hypertension",
    "diabetes",
    "hyperlipidemia",
    "apoe_clean",
    "apoe4_count",
    "apoe4_carrier",
    "age_onset",
    "disease_duration_at_baseline"
]

model_df = model_df.merge(
    demo[demo_features],
    on="patient_id",
    how="left",
    validate="many_to_one"
)

In [22]:
#建立當次 MMSE 時的實際年齡
model_df["age_at_visit"] = (
    model_df["age_first_mmse"]
    + model_df["time_since_baseline_years"]
)

### 合併 CASI

In [24]:
#因為 MMSE 和 CASI 有很多同日資料，建議優先使用：
#同一天 CASI；
#若同日沒有，使用當次 MMSE 前 180 天內最近一次 CASI；
#不使用 MMSE 之後的 CASI，避免未來資料洩漏。

# 先移除 patient_id 缺失，否則無法轉成 int64
model_df = model_df.dropna(subset=["patient_id", "visit_date"]).copy()
casi_for_merge = casi.dropna(subset=["patient_id", "casi_date"]).copy()

# 統一 patient_id 型別
model_df["patient_id"] = model_df["patient_id"].astype("int64")
casi_for_merge["patient_id"] = casi_for_merge["patient_id"].astype("int64")

# 再排序
model_df = model_df.sort_values(
    ["visit_date", "patient_id"]
).reset_index(drop=True)

casi_for_merge = casi_for_merge.sort_values(
    ["casi_date", "patient_id"]
).reset_index(drop=True)

# 合併 CASI
model_df = pd.merge_asof(
    model_df,
    casi_for_merge,
    left_on="visit_date",
    right_on="casi_date",
    by="patient_id",
    direction="backward",
    tolerance=pd.Timedelta(days=180)
)

model_df["days_since_casi"] = (
    model_df["visit_date"] - model_df["casi_date"]
).dt.days

model_df["has_recent_casi"] = (
    model_df["casi_date"].notna().astype(int)
)

### 合併血液資料

In [26]:
#使用當次 MMSE 前 365 天內最近一次抽血，不要用未來抽血
# 清掉合併鍵缺失
model_df = model_df.dropna(
    subset=["patient_id", "visit_date"]
).copy()

blood_for_merge = blood.dropna(
    subset=["patient_id", "blood_date"]
).copy()

# 統一 patient_id 型別
model_df["patient_id"] = pd.to_numeric(
    model_df["patient_id"],
    errors="coerce"
).astype("int64")

blood_for_merge["patient_id"] = pd.to_numeric(
    blood_for_merge["patient_id"],
    errors="coerce"
).astype("int64")

# 確保日期型別一致
model_df["visit_date"] = pd.to_datetime(
    model_df["visit_date"],
    errors="coerce"
)

blood_for_merge["blood_date"] = pd.to_datetime(
    blood_for_merge["blood_date"],
    errors="coerce"
)

# merge_asof 要依時間鍵排序
model_df = model_df.sort_values(
    ["visit_date", "patient_id"]
).reset_index(drop=True)

blood_for_merge = blood_for_merge.sort_values(
    ["blood_date", "patient_id"]
).reset_index(drop=True)

# 合併
model_df = pd.merge_asof(
    model_df,
    blood_for_merge,
    left_on="visit_date",
    right_on="blood_date",
    by="patient_id",
    direction="backward",
    tolerance=pd.Timedelta(days=365)
)

model_df["days_since_blood"] = (
    model_df["visit_date"] - model_df["blood_date"]
).dt.days

model_df["has_recent_blood"] = (
    model_df["blood_date"].notna().astype(int)
)

### 加入缺失指標

In [27]:
important_missing_cols = [
    "casi_total",
    "cdr_sob",
    "apoe4_carrier",
    "hdl",
    "ldl",
    "triglyceride",
    "fasting_glucose",
    "hba1c"
]

for col in important_missing_cols:
    model_df[f"{col}_missing"] = (
        model_df[col].isna().astype(int)
    )

### 建立模型輸入與目標

In [28]:
# 不可以放進模型的欄位
# 以下欄位包含未來資訊或只是識別欄位
exclude_cols = [
    "patient_id",
    "target_date",
    "target_mmse_1y",
    "target_mmse_change_1y",
    "target_followup_days",
    "visit_date",
    "first_mmse_date",
    "previous_visit_date",
    "casi_date",
    "blood_date"
]

#建立特徵與目標
target_col = "target_mmse_1y"

feature_cols = [
    col for col in model_df.columns
    if col not in exclude_cols
]

X = model_df[feature_cols].copy()
y = model_df[target_col].copy()
groups = model_df["patient_id"].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("患者數:", groups.nunique())

X shape: (3193, 63)
y shape: (3193,)
患者數: 723


### 類別與數值欄位

In [29]:
categorical_features = [
    "diagnosis",
    "bio_category",
    "apoe_clean"
]

categorical_features = [
    col for col in categorical_features
    if col in X.columns
]

numeric_features = [
    col for col in X.columns
    if col not in categorical_features
]

print("類別欄位：", categorical_features)
print("數值欄位數：", len(numeric_features))

類別欄位： ['diagnosis', 'bio_category', 'apoe_clean']
數值欄位數： 60


In [33]:
y

0       10.0
1       10.0
2       12.0
3       13.0
4       14.0
        ... 
3188    30.0
3189    24.0
3190     0.0
3191    13.0
3192    13.0
Name: target_mmse_1y, Length: 3193, dtype: float64